# Clase 018 — Boolean masks y fancy indexing

**Parte 0** · VanderPlas cap. 2 §§ 2.6-2.7.

> 🎯 Filtrar, seleccionar y modificar sub-arrays. Vista vs copia.

> ⏱️ ~75 min

## ⚙️ Setup

In [ ]:
import numpy as np
rng = np.random.default_rng(42)

## 1️⃣ Comparaciones → array booleano

Un comparador (`>`, `<`, `==`, `!=`) **no devuelve un bool sino un array de bools**:

In [ ]:
arr = np.array([1, 5, 2, 8, 3, 7])
mask = arr > 3
print('array:', arr)
print('mask :', mask)        # bool array, mismo shape
print('count:', mask.sum())  # cuenta True (los True valen 1)

## 2️⃣ Máscara booleana — filtrado

`arr[mask]` devuelve solo los elementos donde la máscara es True:

In [ ]:
# Lluvia diaria simulada (365 días)
lluvia = rng.gamma(shape=1.5, scale=3, size=365)
# Convierte 60% de los días en "secos" (0)
lluvia[rng.random(365) < 0.6] = 0

print(f'total días     : {len(lluvia)}')
print(f'días lluviosos : {(lluvia > 0).sum()}')
print(f'días >5mm      : {(lluvia > 5).sum()}')
print(f'media (todos)  : {lluvia.mean():.2f} mm')
print(f'media (lluvi.) : {lluvia[lluvia > 0].mean():.2f} mm')

## 3️⃣ Combinar máscaras: `&`, `|`, `~`

⚠️ **NUNCA uses `and`/`or` con arrays** — son escalares. Usa los **bitwise** `&`, `|`, `~` (con paréntesis por precedencia):

In [ ]:
# Lluvia ligera: 1mm <= x <= 10mm
ligeros = lluvia[(lluvia >= 1) & (lluvia <= 10)]
print(f'días lluvia ligera (1-10mm): {len(ligeros)}')

# Lluvia ausente o extrema
raro = lluvia[(lluvia < 1) | (lluvia > 30)]
print(f'días sin lluvia o extremos : {len(raro)}')

# Negación: días sin lluvia
secos = lluvia[~(lluvia > 0)]
print(f'días secos                  : {len(secos)}')

## 4️⃣ Fancy indexing — selección por índices

Pasas un **array de índices** en vez de un slice:

In [ ]:
x = np.arange(10, 100, 10)
print('x:', x)

idx = [0, 3, 5, 8]
print('x[idx]:', x[idx])    # selecciona esos índices

# En matriz: idx por eje
M = np.arange(20).reshape(4, 5)
print('\nM:')
print(M)
print('\nfilas 0 y 2, todas las columnas:')
print(M[[0, 2], :])
print('\nfilas [0,1,2] con columnas [3,1,4] (pares):')
print(M[[0, 1, 2], [3, 1, 4]])   # M[0,3], M[1,1], M[2,4]

## 5️⃣ Modificación in-place — clipping

```python
arr[arr < 0] = 0       # reemplaza negativos por 0
arr[mask] = nuevo_val  # actualización masiva
```

In [ ]:
# Ejemplo: clipping de valores extremos
valores = rng.normal(0, 5, 20).round(2)
print(f'originales: {valores}')

# Clip a [-3, 3]
valores[valores < -3] = -3
valores[valores > 3] = 3
print(f'clipped   : {valores}')

# Equivalente con np.clip
valores2 = rng.normal(0, 5, 20)
np.clip(valores2, -3, 3, out=valores2)

## 6️⃣ ⚠️ Vista vs copia

- **Slicing** (`arr[:5]`, `arr[1:8:2]`) → **vista**. Modificarla modifica el original.
- **Máscara booleana** (`arr[mask]`) → **copia**. Modificarla NO afecta al original.
- **Fancy indexing** (`arr[[0,3,5]]`) → **copia**.

Esto es fuente clásica de bugs.

In [ ]:
arr = np.arange(10)
print(f'original: {arr}')

# Slicing: vista — modifica el original
vista = arr[:5]
vista[0] = 999
print(f'tras vista[0]=999: {arr}  ← cambió!')

# Mask: copia — NO modifica
arr = np.arange(10)
masked = arr[arr > 3]
masked[0] = -1
print(f'tras masked[0]=-1: {arr}  ← sin cambios')

## 7️⃣ `np.where(cond)` sin alternativas — índices donde se cumple

In [ ]:
x = np.array([5, 12, 3, 8, 20, 1])
idx = np.where(x > 5)
print(f'índices donde >5: {idx}')   # tupla con array de índices
print(f'valores         : {x[idx]}')

## ✅ Checklist

- [ ] Filtro con `arr[arr > 0]`
- [ ] Combino con `&`, `|`, `~` (paréntesis)
- [ ] Sé que mask/fancy = copia, slicing = vista
- [ ] Modifico in-place con `arr[mask] = valor`
- [ ] Uso `np.where(cond)` para obtener índices

## 📝 Homework

Ver `README.md`. Análisis de precipitación con máscaras, fancy indexing y demo vista/copia.

## 📖 Definiciones y características

**Boolean mask**

Array de bools del mismo shape que el original. `arr[mask]` extrae solo los elementos donde mask es True. Devuelve un nuevo array (copia, no vista).

**Fancy indexing**

Indexar con array de **enteros** (índices arbitrarios, posiblemente no contiguos). `arr[[0, 3, 5]]` selecciona esas 3 posiciones. Devuelve copia.

**Vista (view) vs copia (copy)**

**Slicing** (`arr[:5]`) → vista (mismo storage, mutarla muta el original). **Mask / fancy** → copia (storage independiente). Fuente del 70% de los bugs sutiles.

**Operadores bitwise vs lógicos**

Para combinar masks: `&`, `|`, `~` (bitwise, vectorizados, elementwise). **NO uses `and`, `or`, `not`** — son escalares Python y dan `ValueError: truth value of an array is ambiguous`.

**`np.where(cond)` 1-arg**

Sin alternativas, devuelve **tupla de arrays de índices** donde se cumple la condición. Diferente a `np.where(cond, a, b)` (ternario).

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `ValueError: The truth value of an array with more than one element is ambiguous` | Usaste `and`/`or`/`not` con arrays. **Fix**: `&`/`|`/`~` con paréntesis: `(a > 0) & (a < 10)` (los paréntesis son obligatorios por precedencia). |
| Modifico `arr[mask]` y el original no cambia | Mask devuelve **copia**. **Fix**: para modificar in-place, asigna a `arr[mask] = nuevo_valor` (no `arr[mask].do_something()`). |
| `arr[idx1, idx2]` con fancy indexing me da algo raro | Si `idx1` e `idx2` son arrays del mismo length, NumPy hace pair-wise: `arr[idx1[i], idx2[i]]` para cada i. Para 'todas las filas idx1 con todas las cols idx2', usa `arr[np.ix_(idx1, idx2)]` o `arr[idx1][:, idx2]`. |
| Slice modifica el original sin querer | `subset = arr[:10]; subset[0] = 99` modifica `arr`. **Fix**: `subset = arr[:10].copy()` si querías independencia. |
| Mask con shape distinto al array | Lanza `IndexError: boolean index did not match indexed array`. **Fix**: asegúrate que la mask tiene el mismo shape (`arr.shape == mask.shape`). |

## ❓ Preguntas frecuentes

**❓ ¿Mask o fancy indexing?**

**Mask** cuando la selección viene de una condición sobre los valores (`arr[arr > 0]`). **Fancy** cuando ya tienes los índices específicos (de un `argsort`, por ejemplo, o de business logic).

**❓ ¿Cómo modifico múltiples elementos con mask?**

`arr[arr < 0] = 0` (clipping). Funciona para asignar escalar a todos los True, o array del mismo tamaño que la cantidad de Trues: `arr[arr < 0] = -arr[arr < 0]` (abs solo donde negativo).

**❓ ¿`arr[idx]` con `idx` como booleano o entero?**

NumPy distingue: bool array del mismo shape → mask; int array → fancy indexing. Lista Python de bools también funciona, pero ojo con `[0, 1, 0, 1]` que puede interpretarse como ints (índices 0 y 1) o bools — usa `np.array(...)` explícito si hay duda.

**❓ ¿`np.where(cond)` o `np.nonzero(cond)`?**

Idénticos cuando `where` se llama con un solo argumento. `nonzero` es más explícito del intent ("dónde NO es 0/False").

**❓ ¿Cómo combino mask con `np.where` ternario?**

`np.where(cond, valor_si_true, valor_si_false)` — ternario vectorizado. La diferencia con `arr[cond] = X`: where construye array nuevo; mask + asignación modifica in-place.

## 🔗 Referencias

- VanderPlas cap. 2 §§ 2.6-2.7
- [Indexing](https://numpy.org/doc/stable/user/basics.indexing.html)

➡️ **Siguiente:** [019 — Ordenamiento y búsqueda](../019-numpy-ordenamiento-y-busqueda/README.md)